In [1]:
from model.Model import model_MSG
# from model.Plotter import plotter_MSG
# from model.coders_linkage import CODERS_pull
from pstats import SortKey, Stats

import os
import sys

# Autoreload modules when changes are applied to them
%load_ext autoreload 
%autoreload all


SyntaxError: invalid syntax (Model.py, line 32)

In [2]:

# Autoreload modules when changiges are applied to them
%load_ext autoreload 
%autoreload all

%reload_ext autoreload
# Set-up
# Sys and OS Operations
os.chdir("../../")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
msg = model_MSG(scenario_name='baseline')
platform, msg_scenario, msg_folder = msg.setup_scenario()

2025-01-29 10:21:59,461  INFO at.ac.iiasa.ixmp.Platform:165 - Welcome to the IX modeling platform!
2025-01-29 10:21:59,465  INFO at.ac.iiasa.ixmp.Platform:166 -  connected to database 'jdbc:hsqldb:file:/Users/muhammadawais/.local/share/ixmp/localdb/default' (user: ixmp)...
Config file read successfully!
####################################################################################################
Config file 'model/scenarios/baseline/config.toml' passed format verification!
####################################################################################################

		Model creation time: 3 Min and 7 Sec 		


In [4]:
from itertools import product

import numpy as np
import pandas as pd
from pycountry import countries, historic_countries

#: Mapping from country names appearing in the database to ISO 3166-1 names.
RENAME_COUNTRY = {
    # Changes required for IEA-EEI related data
    "Korea": "Korea, Republic of",
    "Republic of Korea": "Korea, Republic of",
    "South Korea": "Korea, Republic of",
    "Russia": "Russian Federation",
    # Changes reguired for:
    # data/powerplant_capacity/platts_2017.csv
    # data/powerplant_capacity/carma.csv
    "Antigua & Barbuda": "Antigua and Barbuda",
    "Ascension Island": "Saint Helena, Ascension and Tristan da Cunha",
    "Bosnia-Herzegovina": "Bosnia and Herzegovina",
    "Cape Verde": "Cabo Verde",
    "Channel Islands": "United Kingdom",
    "Congo Dem Rep": "Congo, The Democratic Republic of the",
    "Diego Garcia": "British Indian Ocean Territory",
    "Dutch Caribbean": "Sint Maarten (Dutch part)",
    "England & Wales": "United Kingdom",
    "Iran": "Iran, Islamic Republic of",
    "Laos": "Lao People's Democratic Republic",
    "North Korea": "Korea, Democratic People's Republic of",
    "Northern Marianas": "Northern Mariana Islands",
    "Reunion": "REU",
    "Sao Tome & Principe": "Sao Tome and Principe",
    "Scotland": "United Kingdom",
    "St Helena": "Saint Helena, Ascension and Tristan da Cunha",
    "St Kitts & Nevis": "Saint Kitts and Nevis",
    "St Lucia": "Saint Lucia",
    "St Pierre & Miquelon": "Saint Pierre and Miquelon",
    "St Pierre And Miquelon": "Saint Pierre and Miquelon",
    "St Vincent & Grenadines": "Saint Vincent and the Grenadines",
    "St Vincent And Grenadines": "Saint Vincent and the Grenadines",
    "Swaziland": "Eswatini",
    "Taiwan": "Taiwan, Province of China",
    "Timor Leste": "Timor-Leste",
    "Trinidad & Tobago": "Trinidad and Tobago",
    "Turks & Caicos": "Turks and Caicos Islands",
    "Usa": "United States",
    "Wallis & Futuna": "Wallis and Futuna",
}


def iso_and_region(name, reg):
    """Return (ISO 3166 alpha-3 code, MESSAGEix region) for a country *name*.

    Region names will be returned based on the spatial aggregation of the current
    scenario.

    .. todo:: migrate to a general-purpose utility function in :mod:`message_ix_models`.

    Parameters
    ----------
    name : str
        Country name. This is looked up in the `pycountry
        <https://pypi.org/project/pycountry/#countries-iso-3166>`_ 'name',
        'official_name', or 'common_name' field.
    reg : .Code
        MESSAGEix regions, either ``R11``, ``R14``, ``RCP`` or ``ISR``.

    Returns
    -------
    Series: pandas.Series
    """
    # Maybe map a known, non-standard value to a standard value
    name = RENAME_COUNTRY.get(name.title(), name.title())

    # Use pycountry's built-in, case-insensitive lookup on all fields including name,
    # official_name, and common_name
    alpha_3 = None
    for db in (countries, historic_countries):
        try:
            try:
                alpha_3 = db.lookup(name).alpha_3
            except LookupError:
                alpha_3 = db.search_fuzzy(name)
                if len(alpha_3) > 1:
                    raise ValueError(
                        f"Multiple definitions found for {name}: {alpha_3}"
                    )
                else:
                    alpha_3 = alpha_3[0].alpha_3
        except LookupError:
            # Not found in `db`, e.g. countries; try again
            continue
        else:
            # Found a match; don't try the other db(s)
            break

    # Map from ISO 3166 alpha-3 code to region name
    region = [r.id for r in reg if alpha_3 in r.child]
    if not region:
        # warnings.warn(
        #     "Country {} does not belong to any of the model regions: {}. "
        #     "Please double check.".format(name, [r.id for r in reg])
        # )
        region = [np.nan]

    # Construct a Series and return
    return pd.Series([alpha_3, region[0]], index=["ISO_code", "region"])


def country_to_iso(
    regional_mapping_manual,
    df,
    cntry_col,
    iso_col=None,
    region_id=None,
    msg_reg_col="node_loc",
):
    """Retrieve ISO codes for country names

    If `iso_col` is specified, then the ISO codes will be added into
    that column. Otherwise the country names will be replaced by ISO
    codes.
    If `region_id` is passed, then data as to which corresponding region
    the ISO code belongs to will be added.
    """
    if not iso_col:
        iso_col = cntry_col
    reg_id = "R11" if region_id is None else region_id

    if regional_mapping_manual:
        from .get_codes import get_codes

        nodes = get_codes(f"nodes/{reg_id}")
    else:
        from message_ix_models.model.structure import get_codes

        nodes = get_codes(f"node/{reg_id}")

    regions = nodes[nodes.index("World")].child
    tmp = df[cntry_col].apply(iso_and_region, reg=tuple(regions))
    df[iso_col] = tmp.ISO_code

    if region_id is not None and msg_reg_col is not None:
        df[msg_reg_col] = tmp.region
    return df


def _retrieve_raptis(fil_nm):
    """Read raptis input data

    Parameters
    ----------
    fil_nm : pathlib.Path()
        path and filname incl. extension of data to be read
    """

    drop_cols = [
        "id_unit",
        "COUNTRY",
        "CITY",
        "UNIT",
        "PLANT",
        "STATUS",
        "YEAR",
        "Notes",
        "max_mean_annual_nominator_fuel_consumption",
    ]
    df = pd.read_excel(fil_nm, skiprows=range(0, 26)).drop(drop_cols, axis=1)
    return df


def _retrieve_platts(fil_nm):
    """Read platts input data

    Parameters
    ----------
    fil_nm : pathlib.Path()
        path and filname incl. extension of data to be read
    """
    drop_cols = [
        "ALTFUEL",
        "TURBMFR",
        "TURBTYPE",
        "GENMFR",
        "GENTYPE",
        "SFLOW",
        "SSSMFR",
        "TURBMFR",
        "TURBTYPE",
        "AE",
        "CONSTRUCT",
    ]
    df = pd.read_csv(fil_nm, encoding="utf-8", low_memory=False).drop(drop_cols, axis=1)

    # Remove incorrect capacity for NORTH Korea.
    # The 5MW nuclear reacotr is not a power station and should be removed.
    df = df.loc[
        ~((df.FUEL == "UR") & (df.COUNTRY == "NORTH KOREA") & (df.STATUS == "OPR"))
    ]

    return df


def _retrieve_carma(fil_nm):
    """Read carma input data

    Parameters
    ----------
    fil_nm : pathlib.Path()
        path and filname incl. extension of data to be read
    """
    drop_cols = [
        "plant_id",
        "orispl",
        "crd",
        "fuel2",
        "int_2004",
        "hr_2004",
        "dis",
        "pmover",
        "dfuel",
        "sfg",
        "fbt",
        "lis",
        "aci",
        "super",
        "ccap",
        "bustype",
        "electype",
        "sb",
        "mw_2009",
        "mw_2004",
        "co2_2009",
        "mwh_2009",
        "co2_2004",
        "mwh_2004",
    ]
    df = pd.read_csv(fil_nm).drop(drop_cols, axis=1)
    return df


def _merge_data(platts, carma, raptis):
    """Mege various input data files.

    Parameters
    ----------
    platts : pandas.DataFrame()
        platts input data
    carma : pandas.DataFrame()
        carma input data
    raptis : pandas.DataFrame()
        raptis input data
    """

    # Merge carma data onto platts data using "LOCATIONID"
    df = pd.merge(
        left=platts,
        right=carma,
        how="left",
        left_on="LOCATIONID",
        right_on="locationid",
    )

    # Merge raptis-data on platts-data using "UNITID"
    df = pd.merge(
        left=df, right=raptis, how="left", left_on="UNITID", right_on="UNITID"
    ).rename(columns={"Location": "location"})

    return df


def _assign_location(df):
    """Merging of lat-lon data from carma and raptis.

    Case 1 Use Raptis if:
    - Raptis lat/long is present
    - Carma latitude/longitude not present

    Case 2 Use Carma if:
    - Carma latitude/longitude present
    - Raptis lat/long not present

    Case 3
    - carma latitude/longitude present && raptis lat/long  present
    - if 'location' == 'exact', use raptis lat/long
    - otherwise - need preference of either carma or raptis

    Parameters
    ----------
    df : pandas.DataFrame()
        dataframe containing merged input data
    """

    # Add new empty columns which will be used to populate the
    # the latitude and longitude
    df["latC"] = np.nan
    df["lonC"] = np.nan
    df["locationC"] = np.nan

    # Case 1 Apply Raptis
    # Columns "lat", "long", "location" associated with Raptis
    df.loc[(df["lat"].notnull()) & (df["latitude"].isnull()), "latC"] = df.loc[
        (df["lat"].notnull()) & (df["latitude"].isnull()), "lat"
    ]
    df.loc[(df["long"].notnull()) & (df["longitude"].isnull()), "lonC"] = df.loc[
        (df["long"].notnull()) & (df["longitude"].isnull()), "long"
    ]
    df.loc[(df["long"].notnull()) & (df["longitude"].isnull()), "locationC"] = df.loc[
        (df["long"].notnull()) & (df["longitude"].isnull()), "location"
    ]

    # Case 2 Apply Carma
    # Columns "latitude", "longitude" are associated with Carma
    df.loc[(df["lat"].isnull()) & (df["latitude"].notnull()), "latC"] = df.loc[
        (df["lat"].isnull()) & (df["latitude"].notnull()), "latitude"
    ]

    df.loc[(df["long"].isnull()) & (df["longitude"].notnull()), "lonC"] = df[
        "longitude"
    ][(df["long"].isnull()) & (df["longitude"].notnull())]
    df.loc[(df["long"].isnull()) & (df["longitude"].notnull()), "locationC"] = (
        "carmageo"
    )

    # Case 3 Exact
    df.loc[
        (df["lat"].notnull())
        & (df["latitude"].notnull())
        & (df["location"] == "exact"),
        "latC",
    ] = df.loc[
        (df["lat"].notnull())
        & (df["latitude"].notnull())
        & (df["location"] == "exact"),
        "lat",
    ]

    df.loc[
        (df["long"].notnull())
        & (df["longitude"].notnull())
        & (df["location"] == "exact"),
        ["lonC", "locationC"],
    ] = np.array(
        [
            df["long"][
                (df["long"].notnull())
                & (df["longitude"].notnull())
                & (df["location"] == "exact")
            ],
            df["location"][
                (df["long"].notnull())
                & (df["longitude"].notnull())
                & (df["location"] == "exact")
            ],
        ]
    ).transpose()
    return df


def _georeference(df, method="raptis"):
    """Add georeference to data.

    Parameters
    ----------
    df : pandas.DataFrame()
        dataframe containing merged input data
    method : str (default="raptis")
        method applied for the georeferencing data
    """

    if method == "carmageo":
        lat_idx = "latitude"
        long_idx = "longitude"
        val = "carmageo"
    else:
        lat_idx = "lat"
        long_idx = "long"
        val = "georeferenced"

    df.loc[
        (df["lat"].notnull())
        & (df["latitude"].notnull())
        & (df["location"] == "georeferenced"),
        "latC",
    ] = df.loc[
        (df["lat"].notnull())
        & (df["latitude"].notnull())
        & (df["location"] == "georeferenced"),
        lat_idx,
    ]

    df.loc[
        (df["long"].notnull())
        & (df["longitude"].notnull())
        & (df["location"] == "georeferenced"),
        ["lonC", "locationC"],
    ] = np.array(
        [
            df[long_idx][
                (df["long"].notnull())
                & (df["longitude"].notnull())
                & (df["location"] == "georeferenced")
            ],
            val,
        ],
        dtype=object,
    ).transpose()
    return df


def _assign_msg_attribute(df, path_capacity_platts_info):
    """Assign aggregate groupings to powerplants

    The PLATTS fuel and unit-type information as well are used
    first aggregated to MESSAGEix compatible groupings. Subsequently,
    cooling groups are assigned.

    Parameters
    ----------
    df : pd.DataFrame()
        dataframe containing data to which colling type will be added
    """

    xlsx = pd.ExcelFile(path_capacity_platts_info)

    # Assign new columns and dummy data
    df = df.assign(
        utype_group="",
        fuel_group="",
        cool_group="",
        cool_group_msg="",
        utype_group_msg="",
        utype_group_msg2="",
    )

    # For each fuel, utype and cooling type, assign to a group in new column
    # Unit types must be processed before cool_groups

    # -----------------------------------
    # For each fuel assign to a aggreagte
    # -----------------------------------

    MC_fuels = pd.read_excel(xlsx, sheet_name="FUEL")

    grp = [["fuel_group", "fuel_groups"]]
    for f, g in product(MC_fuels.WEPP_fuel, grp):
        df.loc[df.FUEL_x == f, g[0]] = MC_fuels.loc[
            MC_fuels.WEPP_fuel == f, g[1]
        ].item()

    # ----------------------------------------
    # For each unit-type assign to a aggreagte
    # ----------------------------------------

    MC_utypes = pd.read_excel(xlsx, sheet_name="UTYPE")

    grp = [
        ["utype_group", "utype_groups"],
        ["cool_group", "cool_groups"],
        ["utype_group_msg", "utype_msg"],
        ["utype_group_msg2", "utype_msg2"],
        ["cool_group_msg", "cool_groups"],
    ]
    for u, g in product(MC_utypes.WEPP_utype, grp):
        df.loc[df.UTYPE_x == u, g[0]] = MC_utypes.loc[
            MC_utypes.WEPP_utype == u, g[1]
        ].item()

    # -----------------------------------------------
    # For each unit-type assign to a aggreagte
    # Unit types must be processed before cool_groups
    # -----------------------------------------------

    MC_cools = pd.read_excel(xlsx, sheet_name="COOL")

    grp = [["cool_group", "cool_groups"], ["cool_group_msg", "cool_groups_msg"]]

    # Copy over raptis power plant cool data
    df.cool_group = df.COOL_complete

    for c, g in product(MC_cools.WEPP_cool, grp):
        df.loc[df.COOL_complete == c, g[0]] = MC_cools.loc[
            MC_cools.WEPP_cool == c, g[1]
        ].item()

    # list of aggregate utype, fuel and cool groups
    MC_groups = pd.read_excel(xlsx, sheet_name="Groups")

    # combination table of water use factors for each group
    MC_UFC = pd.read_excel(xlsx, sheet_name="UFC-combo")

    return (df, MC_groups, MC_UFC)


def _assign_msg_technology(df):
    # Combine fuel and unit into single code
    # The difference between "utype_group_msg" and "utype_group_msg2"
    # is that the first differentiates in more detail between
    # "st" (steam_turbines) where as the latter is more specifically
    # adapted to the MESSAGEix generic definitions. The former though
    # is needed in order to make fuel-specific adjustments.
    df["msg_combo"] = df.fuel_group + "_" + df.utype_group_msg
    df["technology"] = df.fuel_group + "_" + df.utype_group_msg2

    # The following routines make use of the more detailed data of
    # "utype_group_msg" and undertake further changes to the final
    # technology allocation in MESSAGEix.

    # ----------------
    # Coal powerplants
    # ----------------

    # Assign "coal_ppl_u" if:
    # - msg_combo coal-steam turbine
    # - STYPE is subcritical
    # - NO FGDMFR available
    df.loc[
        ((df.msg_combo == "coal_st") & (df.STYPE == "SUBCR") & (df.FGDMFR.isnull())),
        "technology",
    ] = "coal_ppl_u"

    # Assign "coal_ppl_u" if:
    # - msg_combo coal-steam turbine
    # - STYPE is supercritical
    # - FGDMFR available
    df.loc[
        (
            (df.msg_combo == "coal_st")
            & (df.STYPE == "SUPERC")
            # & (df.FGDMFR.notnull()) # excluded in v3
            # & (df.FGDMFR != "N/A") # excluded in v3
        ),
        "technology",
    ] = "coal_adv"

    # Assign "igcc" if:
    # - msg_combo is coal_cc
    df.loc[((df.msg_combo == "coal_cc")), "technology"] = "igcc"

    # -------------------
    # Biomass powerplants
    # -------------------

    # Assign "bio_istig" if:
    # - msg_combo is bio_cc
    df.loc[((df.msg_combo == "bio_cc")), "technology"] = "bio_istig"

    # Assign "bio_ppl" if:
    # - msg_combo is bio_ct
    df.loc[((df.msg_combo == "bio_ct")), "technology"] = "bio_ppl"

    # Assign "bio_ppl" if:
    # - msg_combo is bio_pv_ppl
    df.loc[((df.msg_combo == "bio_pv_ppl")), "technology"] = "bio_ppl"

    # -------------------------------
    # Light and heavy oil powerplants
    # -------------------------------

    # Assign "foil_ppl" if:
    # - msg_combo is foil_ct, foil_cc, foil_st, foil_pv_ppl
    df.loc[
        (
            (df.msg_combo == "foil_ct")
            | (df.msg_combo == "foil_cc")
            | (df.msg_combo == "foil_st")
            | (df.msg_combo == "foil_pv_ppl")
        ),
        "technology",
    ] = "foil_ppl"

    # Assign "loil_ppl" if:
    # - msg_combo is loil_ct, foil_st
    df.loc[
        ((df.msg_combo == "loil_ct") | (df.msg_combo == "loil_st")), "technology"
    ] = "loil_ppl"

    # -----
    # Solar
    # -----

    # Assign "solar_pv_ppl" if:
    # - msg_combo is solar_pv_ppl
    df.loc[((df.msg_combo == "solar_pv_ppl")), "technology"] = "solar_pv_ppl"

    # Assign "solar_th_ppl" if:
    # - msg_combo is solar_ and "fueltype" is "COL" (Solar collector including
    #   solar dish arrays and solar towers and solar troughs)
    # - msg_combo is "solar_ppl", "solar_st", "solar_ct"
    df.loc[((df.msg_combo == "solar_") & (df.FUELTYPE == "COL")), "technology"] = (
        "solar_th_ppl"
    )
    df.loc[((df.msg_combo == "solar_ppl")), "technology"] = "solar_th_ppl"
    df.loc[((df.msg_combo == "solar_st")), "technology"] = "solar_th_ppl"
    df.loc[((df.msg_combo == "solar_ct")), "technology"] = "solar_th_ppl"

    # ----------
    # Geothermal
    # ----------

    # geothermal & NCN
    # df.loc[((df.msg_combo == 'geo_ppl') & (df.UTYPE_x == 'ST/S')),
    #        'technology'] = 'geo_hpl'

    # --------------
    # Municpal Waste
    # --------------

    # Assign "msw_ppl" if:
    # - msg_combo is msw_cc
    df.loc[((df.msg_combo == "msw_cc")), "technology"] = "msw_ppl"

    # ----------
    # Hydropower
    # ----------
    # Assign "hyrdo_hc" if:
    # - fuel_group is water and the capacity exceeds 15MW
    df.loc[((df.fuel_group == "wat") & (df.MW_x > 15)), "technology"] = "hydro_hc"

    # Assign "hyrdo_lc" if:
    # - fuel_group is water and the capacity is equal-to or below 15MW
    df.loc[((df.fuel_group == "wat") & (df.MW_x <= 15)), "technology"] = "hydro_lc"

    # -------
    # Nuclear
    # -------

    # Assign "nuc_lc" if the fuel-types are defined as follows:
    # - MAG - GenI. - MAGNOX gas-cooled reactor (Mostly retired, 1 operational: 80's)
    # - OMR - GenI. Organic moderator reactor (Retired)
    # - AGR - GenII. - Advanced gas-cooled reactor (Operational: 70s, 80s, UK-Only)
    # - BWR - GenII. - Boiling water reactor (Operational: 60s-90s)
    # - CANDU - GenII. - CANDU (CANadian Deuterium Uranium) pressurized heavy water
    #                    reactor (Operational: 70 to 00's, Canada)
    # - GCR - GenII. - Gas-cooled reactor (other) (Retired or Cancelled)
    # - HWGCR - GenII. - Heavy water gas-cooled reactor (Retired)
    # - HWR - Heavy water reactor (Retired)
    # - LWR - GenII. - Light water reactor (Planned)
    # - RBMK - GenII. - Pressurized water reactor (Operational:60's, 70's, 80's)

    for f in [
        "MAG",
        "OMR",
        "AGR",
        "BWR",
        "CANDU",
        "GCR",
        "HWGCR",
        "HWR",
        "LWR",
        "RBMK",
    ]:
        df.loc[(df.FUELTYPE == f), "technology"] = "nuc_lc"

    # If the fuel-types are defined as follows:
    # - PHWR - GenII. - Pressurized heavy water reactor (Operational: 70s to today;
    #                   all greater 2000 GenIII.)
    # - PWR - GenII. - Pressurized water reactor (Operational: 80s to today;
    #                  all greater 2000 GenIII.)
    # All those installed prior to 2000 are assinged to "nuc_lc"
    # All those installed as of 2000 are assinged to "nuc_hc"

    for f in ["PHWR", "PWR"]:
        df.loc[((df.FUELTYPE == f) & (df.YEAR < 2000)), "technology"] = "nuc_lc"
        df.loc[((df.FUELTYPE == f) & (df.YEAR >= 2000)), "technology"] = "nuc_hc"

    # Assign "nuc_hc" if the fuel-types are defined as follows:
    # - ABWR - GenIII. - Advanced boiling water reactor (Standby or Planned)
    # - APWR - GenIII. - Advanced pressurized water reactor
    #          (Planned or Construction: 2016 Onwards)
    # - EPR - GenIII. - Evolutionary or European power reactor (Areva APWR design)
    #         (Cancelled)
    # - FBR - GenIII/IV. - Fast breeder reactor (Operational or Planned: 201X)
    # - HTGR - GenIII. - High temperature gas-cooled reactor (Construction or Planned)
    # - PBMR - GenIV. - Pebble bed modular reactor (Cancelled)
    # - SMR - GenIV. Small modular reactor (Construction)
    # - SVBR - GenIV. - Lead-bismuth fast reactor, Russian submarine-derived reactor
    #          design (Planned)

    for f in ["ABWR", "APWR", "EPR", "FBR", "FBR", "HTGR", "PBMR", "SMR", "SVBR"]:
        df.loc[(df.FUELTYPE == f), "technology"] = "nuc_hc"

    # -----------------
    # Unkown fuel-types
    # -----------------

    # Assign "gas_ppl" if:
    # - fueltype is UNK (unknown fuel-type)
    # - UTYPE is GT/S
    df.loc[
        ((df.FUEL_x == "UNK") & (df.UTYPE_x == "GT/S")), ["fuel_group", "technology"]
    ] = ["gas", "gas_ppl"]

    # Assign "coal_ppl" if:
    # - fueltype is UNK (unknown fuel-type)
    # - UTYPE is ST/S or ST
    df.loc[
        ((df.FUEL_x == "UNK") & ((df.UTYPE_x == "ST/S") | (df.UTYPE_x == "ST"))),
        ["fuel_group", "technology"],
    ] = ["coal", "coal_ppl"]
    return df


def _infer_cooling(df):
    """Inferred cooling system assignment."""

    # STEAM + NUCLEAR
    # If cool_group is empty and fuel==nuc and utype is ST
    # assign OT cooling (once-through) and 'ot_saline'
    df.loc[
        (df["cool_group"].isnull())
        & (df["fuel_group"] == "nuc")
        & (df["utype_group"] == "ST"),
        ["cool_group", "cool_group_msg", "Cooling_info_origin"],
    ] = ["OT", "ot_saline", "Infer"]

    # STEAM + not NUC, assume CL (closed-loop)
    df.loc[
        (df["cool_group"].isnull())
        & (df["fuel_group"] != "nuc")
        & (df["utype_group"] == "ST"),
        ["cool_group", "cool_group_msg", "Cooling_info_origin"],
    ] = ["CL", "cl_fresh", "Infer"]

    # Combined Cycle, assume CL (closed-loop)
    df.loc[
        (df["cool_group"].isnull()) & (df["utype_group"] == "CC"),
        ["cool_group", "cool_group_msg", "Cooling_info_origin"],
    ] = ["CL", "cl_fresh", "Infer"]

    # If cool_group == NCN and cooling_info_origin is empty,
    # make it == NCN
    df.loc[
        (df["cool_group"] == "NCN") & (df["Cooling_info_origin"].isnull()),
        ["cool_group", "cool_group_msg", "Cooling_info_origin"],
    ] = ["NCN", "NCN", "NCN"]
    return df


def _add_wateruse(df, MC_groups, MC_UFC):
    df["WWf"] = 0.0
    df["WCf"] = 0.0
    df["LF"] = 0.0
    df["WW"] = 0.0
    df["WC"] = 0.0

    for fuel, utype, cool in product(
        MC_groups.fuel_groups, MC_groups.utype_groups, MC_groups.cool_groups
    ):
        # If statement to ensure that row is not empty and prevent error
        if (
            len(
                MC_UFC.WWf[
                    (MC_UFC.utype_groups == utype)
                    & (MC_UFC.fuel_groups == fuel)
                    & (MC_UFC.cool_groups == cool)
                ]
            )
            != 0
        ):
            # Assign correct cooling factors WWf and WCf
            df.WWf[
                (df.utype_group == utype)
                & (df.fuel_group == fuel)
                & (df.cool_group == cool)
            ] = MC_UFC.WWf[
                (MC_UFC.utype_groups == utype)
                & (MC_UFC.fuel_groups == fuel)
                & (MC_UFC.cool_groups == cool)
            ].item()
            df.WCf[
                (df.utype_group == utype)
                & (df.fuel_group == fuel)
                & (df.cool_group == cool)
            ] = MC_UFC.WCf[
                (MC_UFC.utype_groups == utype)
                & (MC_UFC.fuel_groups == fuel)
                & (MC_UFC.cool_groups == cool)
            ].item()
            # Load factors
            df.LF[
                (df.utype_group == utype)
                & (df.fuel_group == fuel)
                & (df.cool_group == cool)
            ] = MC_UFC.LF[
                (MC_UFC.utype_groups == utype)
                & (MC_UFC.fuel_groups == fuel)
                & (MC_UFC.cool_groups == cool)
            ].item()

    df["WW"] = df.MW_x * df.LF * 8766.0 * df.WWf
    df["WC"] = df.MW_x * df.LF * 8766.0 * df.WCf
    return df


def _write_data(
    df,
    write_format,
    write_short=False,
    write_long=False,
):
    file_name = "historical_capacity_data"

    if write_long:
        if write_format == "xlsx":
            df.to_excel(path / f"{file_name}.xlsx", index=False)
        else:
            df.to_csv(path / f"{file_name}.csv", index=False)

    if write_short:
        df = df[
            [
                "UNIT",
                "MW_x",
                "STATUS",
                "YEAR",
                "UTYPE_x",
                "FUEL_x",
                "CITY",
                "COUNTRY",
                "ELECTYPE",
                "chp",
                "ISO",
                "node_loc",
                "utype_group",
                "technology",
            ]
        ]
        if write_format == "xlsx":
            df.to_excel(path / f"{file_name}_short.xlsx", index=False)
        else:
            df.to_csv(path / f"{file_name}_short.csv", index=False)


# def _add_hydrobasins(wbraps, path, write_additional_file=False):
#     """Add hydrobasins data (points into shapes)"""
#     import geopandas as gpd
#     import glob2
#     from geopandas.tools import sjoin
#     from shapely.geometry import Point
#
#     # If opening from file:
#     # wbraps = pd.read_csv(path / 'powerplants' / 'IIASAPP',
#     #                      3.6 / 'IIASAPP.csv',
#     #                      encoding='cp1252')
#     # Drop the rows with no lat lon info
#     df = wbraps.dropna(subset=["latC", "lonC"], how="any")
#
#     # These are the columns NOT to drop
#     drcols = df.columns.drop(["latC", "lonC", "Unnamed: 0"])
#     df.drop(drcols, axis=1, inplace=True)
#
#     df["geometry"] = df.apply(lambda z: Point(z.lonC, z.latC), axis=1)
#
#     levels = range(1, 5)
#     PointsGeodataframe = gpd.GeoDataFrame(df)
#     tpmerged = []
#     spmerged = []
#     for level in levels:
#         print("Level: " + str(level))
#         shapes = glob2.glob(
#             "P:\\is-wel\\shared_data_sources\\raw_data"
#             + "\\hydroBASINS\\*merged*\\**.shp"
#         )
#         ctlist = []
#         ct = 0
#         # Shapes are the regions
#         for shapefile in shapes:
#             cts = "ct" + str(ct)
#             PolygonsGeodataframe = gpd.GeoDataFrame.from_file(shapefile)
#
#             # Trim data in Polygons
#             PolygonsGeodataframe = PolygonsGeodataframe[["geometry", "HYBAS_ID"]]
#
#             PointsGeodataframe.crs = PolygonsGeodataframe.crs
#
#             spmerged = sjoin(
#                 PointsGeodataframe, PolygonsGeodataframe, how="left", op="within"
#             )
#             if ct == 0:
#                 tpmerged = spmerged
#                 tpmerged.rename(index=str, columns={"HYBAS_ID": cts}, inplace=True)
#                 ctlist = [cts]
#             else:
#                 tpmerged[cts] = spmerged.HYBAS_ID.values
#                 ctlist.append(cts)
#
#             ct = ct + 1
#
#         PointsGeodataframe["HYBAS_ID_" + str(level)] = (
#             tpmerged[ctlist].max(axis=1).values
#         )
#
#     # Merge PointsGeodataframe into original table (dfo)
#     colslist = [
#         "Unnamed: 0",
#         "HYBAS_ID_1",
#         "HYBAS_ID_2",
#         "HYBAS_ID_3",
#         "HYBAS_ID_4",
#         "HYBAS_ID_5",
#         "HYBAS_ID_6",
#     ]
#     bm = pd.merge(
#         wbraps,
#         PointsGeodataframe[colslist[0 : levels[-1] + 1]],
#         how="left",
#         left_on="Unnamed: 0",
#         right_on="Unnamed: 0",
#     )
#     # bm = pd.merge(wbraps, PointsGeodataframe[['Unnamed: 0','HYBAS_ID_1']],
#     #               how='left', left_on='Unnamed: 0', right_on='Unnamed: 0')
#     # bm = pd.merge(wbraps, PointsGeodataframe[['Unnamed: 0','HYBAS_ID_1']],
#     #               how='left', left_on='Unnamed: 0', right_on='Unnamed: 0')
#
#     # Save raw output
#     bm.to_csv(path / "powerplants" / "IIASAPP" / 3.6 / "IIASAPP_HB.csv")
#
#     if write_additional_file:
#         # Drop columns
#         tmp = PointsGeodataframe.columns.drop(["latC_y", "lonC_y", "geometry"])
#         tmp.to_csv(path / "powerplants" / "IIASAPP" / 3.6 / "IIASAPP_HYB_clean.csv")
#
#         # Drop values with no lat-lon if nec
#         tmp = bm.dropna(subset=["latC", "lonC"], how="any")
#         tmp.to_csv(path / "powerplants" / "IIASAPP" / 3.6 / "IIASAPP_clean.csv")



In [5]:
regional_mapping_manual=False
write_format="csv"
add_hydrobasins=False
region_id=None
return_data=False
write_long=True
write_short=False
private_data_path=None
path_capacity_carma='/Users/muhammadawais/Documents/GitHub/message-ix-canada/data/powerplant_capacity/carma.csv'
path_capacity_platts='/Users/muhammadawais/Documents/GitHub/message-ix-canada/data/powerplant_capacity/platts_2021.csv'
path_capacity_platts_info='/Users/muhammadawais/Documents/GitHub/message-ix-canada/data/powerplant_capacity/platts_2021_info.xlsx'
path_capacity_raptis='/Users/muhammadawais/Documents/GitHub/message-ix-canada/data/powerplant_capacity/raptis_2016.xlsx'


In [6]:

"""Combine the powerplant data from different sources.
The puprose of the script is to merge powerplant data from different sources.
The original script was written by Ed Byers.
The original script can be found at
`/p/ene.model/data/powerplants/IIASAPP/v3.6/Merge_WEPP_Raptis_CARMA_v3.6.py`

The script combines data from three sources:
1. Platts (2021)
2. Raptis (2016)
3. Carma (2017)

Parameters
----------
add_hydrobasins : boolean (default=False)
    Option for adding hydrobasin data.
    NOTE: This function has not been tested/used. A warning has been added.
"""

# Specify variables
global path

# Specify path where necessary input data can be found
# path = private_data_path("model", "powerplant_capacity")
path = '/Users/muhammadawais/Documents/GitHub/message-ix-canada/data/powerplant_capacity'

# --------------------------
# Step1. Retrieve input data
# --------------------------
raptis = _retrieve_raptis(path_capacity_raptis)
platts = _retrieve_platts(path_capacity_platts)
carma = _retrieve_carma(path_capacity_carma)


In [7]:
# -----------------
# Step2. Merge data
# -----------------

df = _merge_data(platts, carma, raptis)

# ------------------
# Step3. Format data
# ------------------

# Assign Latitude and Longitude
# using either carma or raptis
df = _assign_location(df)

# Assign Latitude and Longitude
# using georeference data
df = _georeference(df)

In [8]:
# Assign ISO-codes and MESSAGE-region-id
df = country_to_iso(
    regional_mapping_manual,
    df,
    "COUNTRY",
    "ISO",
    region_id=region_id, 
)

SubdivisionHierarchy(code='KI-L', country_code='KI', name='Line Islands', parent_code=None, type='Group of islands (20 inhabited islands)')
SubdivisionHierarchy(code='KI-L', country_code='KI', name='Line Islands', parent_code=None, type='Group of islands (20 inhabited islands)')
SubdivisionHierarchy(code='GB-NIR', country_code='GB', name='Northern Ireland', parent_code=None, type='Province')
SubdivisionHierarchy(code='GB-NIR', country_code='GB', name='Northern Ireland', parent_code=None, type='Province')
SubdivisionHierarchy(code='GB-NIR', country_code='GB', name='Northern Ireland', parent_code=None, type='Province')
SubdivisionHierarchy(code='GB-NIR', country_code='GB', name='Northern Ireland', parent_code=None, type='Province')
SubdivisionHierarchy(code='GB-NIR', country_code='GB', name='Northern Ireland', parent_code=None, type='Province')
SubdivisionHierarchy(code='GB-NIR', country_code='GB', name='Northern Ireland', parent_code=None, type='Province')
SubdivisionHierarchy(code='GB-

In [9]:
# Assign message attributess
df, MC_groups, MC_UFC = _assign_msg_attribute(df, path_capacity_platts_info)

In [10]:
# Derive message technology
df = _assign_msg_technology(df)

# Inferred cooling system assignment
df = _infer_cooling(df)

# Assign water use
df = _add_wateruse(df, MC_groups, MC_UFC)

In [11]:
df.to_excel( "power_plant_capacity.xlsx", index=False)

In [12]:
df = pd.read_excel( "power_plant_capacity.xlsx")

In [13]:
df = df[df['ISO'] =='CAN']

In [14]:
df = df[[ "UNIT",
                "MW_x",
                "STATUS",
                "YEAR",
                "UTYPE_x",
                "FUEL_x",
                "CITY",
                "COUNTRY",
                "ELECTYPE",
                "chp",
                "ISO",
                "STATE",
                "CITY",
                "utype_group",
                "technology",
            ]]

In [15]:
# Group by STATE, technology, and YEAR, then sum the other columns
grouped_df = df.groupby(["STATE", "technology", "YEAR"], as_index=False).sum()

In [16]:
df.to_csv('historical_capacity_can_disagregated.csv')

In [17]:
import pandas as pd

In [18]:
grouped_df = pd.read_csv('/Users/muhammadawais/Documents/GitHub/message-ix-canada/historical_capacity_by_province.csv')

In [19]:
df = pd.read_csv('/Users/muhammadawais/Documents/GitHub/message-ix-canada/data/powerplant_capacity/historical_capacity_can_disagregated.csv')

In [20]:
scen=msg_scenario
# region_id="R11"
read_csv=False
compile_capacity_data_only=None
platts_exclude_tecs=False
remove_orig=True
verbose=True
iaea=False
irena=False
add_total_capacity_lo=None
add_total_capacity_lo_tec="all"

In [21]:
from model.utlities.get_historical_years import main as get_historical_years


In [22]:

# Conversion factor from (MW) to (GW)
# Original data is provided in MW and needs to be converted
# to the model units GW
unit_conv = 0.001


def aggregate_capacity_data(
    df,
    scen,
    # planned="Construction",
    planned=False,
    fuel_remove=["UNK", "METHANOL", "HELIUM"],
    country_model=False,
):
    """Calibrates historical power-plant capacities.

    The merged data from "PLATTS", "RAPTIS" and "CARMA" at the country
    level is aggregated across regions (optional) and grouped into bins
    which correspond the the model specific historic timeperiods
    i.e. all time-steps < `firstmodelyear`.

    Parameters
    ----------
    df : pd.DataFrame
        raw data generated using `compile_capacity_data`
    scen : message_ix.Scenario
        scenario for which capacity calibration should be carried out
    planned : boolean
        option whther to include powerplants with a planned status.
    fuel_remove : list (default=["UNK", "METHANOL", "HELIUM"])
        a list of fuel-types based on PLATTS definition
        (see FUEL in `data/powerplant_capacity/platts_2017_info.xlsx`)
    country_model : boolean/list
        If false, then it is assumed the scenario corresponds to a global
        version. Otherwise a list of ISO codes can be provided and data will
        be filtered for these.
    """

    # Define which operational status to drop;
    # "else" drops everythin except operational.
    if planned is True:
        status_drop = ["RET", "CAN", "STN", "DAC", "UNK", "DEF", "DEL"]
    elif planned == "construction":
        status_drop = ["RET", "CAN", "STN", "PLN", "DAC", "UNK", "DEF", "DEL"]
    else:
        status_drop = ["RET", "CAN", "STN", "PLN", "DAC", "UNK", "DEF", "CON", "DEL"]

    # Include only the powerplants with the correct status
    df = df.loc[~df.STATUS.isin(status_drop)]

    # Drop unallocated technologies
    df.dropna(subset=["technology"], inplace=True)

    # Drops all technologies which are not included in the scenario
    df = df.loc[df.technology.isin(scen.set("technology"))]

    # Drop all fuels that should be removed
    df.drop(df.loc[df["FUEL_x"].isin(fuel_remove)].index, inplace=True)

    # Drop all unassigned YEARs
    # Several powerplants have no "YEAR" defined, meaning that it is unclear
    # when these powerplants were built.
    df.drop(df[np.isnan(df.YEAR)].index, inplace=True)

    # Create bins, representing the scenario specifc historic periods.
    # Capacity is then assigned to these bins.
    bins = get_historical_years(scen, year_min=min(scen.set("year")))
    bins = bins + [scen.firstmodelyear]
    mid = [int((a + b) / 2) for a, b in zip(bins, bins)][1:]
    df["bins"] = pd.cut(df["YEAR"], bins=bins).cat.rename_categories(mid)

    # Data will be summed across the bins, and divided by the period-duration
    # in order to derive an annual value. MW get converted to GW.
    groups = (
        df[["STATE", "technology", "bins", "MW_x"]]
        .groupby(["STATE", "technology", "bins"], observed=False)
        .sum()
    )
    period_duration = (
        scen.par("duration_period", filters={"year": bins})
        .rename(columns={"year": "bins"})
        .set_index("bins")
    )
    data = groups["MW_x"] / period_duration["value"] * unit_conv

    # Format accordingly for parameter `historical_new_capacity`
    data = (
        data.reset_index()
        .rename(columns={"STATE": "node_loc", "bins": "year_vtg", 0: "value"})
        .assign(unit="GW")
        .sort_values(by=["node_loc", "technology", "year_vtg"])
    )

    # if not country_model:
    #     # Create a dictionary from the original dataframe mapping countries to
    #     # message regions and apply
    #     tdict = (
    #         df[["COUNTRY", "node_loc"]]
    #         .drop_duplicates()
    #         .sort_values(by="node_loc")
    #         .dropna()
    #         .set_index("COUNTRY")
    #         .to_dict()["node_loc"]
    #     )
    #     data["node_loc"] = data["node_loc"].map(tdict)
    #     idx = ["technology", "node_loc", "unit", "year_vtg"]
    #     data = data.groupby(idx).sum().reset_index().dropna(how="any")
    # else:
    #     # Create a dictionary from the original dataframe and rename countries
    #     # to ISO-3.
    #     tdict = (
    #         df[["COUNTRY", "ISO"]]
    #         .drop_duplicates()
    #         .sort_values(by="node_loc")
    #         .dropna()
    #         .set_index("COUNTRY")
    #         .to_dict()["ISO"]
    #     )

    #     # Ensure country model is a list
    #     if type(country_model) != "list":
    #         country_model = [country_model]
    #     data = data.loc[data.ISO.isin(country_model)]

    return data

In [23]:
import numpy as np

In [24]:
df = aggregate_capacity_data(df, scen)

In [25]:
df.rename(columns = {'STATE':'node_loc'}, inplace = True)

In [26]:
df.columns

Index(['node_loc', 'technology', 'year_vtg', 'value', 'unit'], dtype='object')

In [27]:
region_rename_map = {
    'NL': 'NewfoundlandandLabrador',
    'BC': 'BritishColumbia',
    'PE': 'PrinceEdwardIsland',
    'NT': 'NorthwestTerritories',
    'YK': 'Yukon',
    'NS': 'NovaScotia',
    'NB': 'NewBrunswick',
    'ON': 'Ontario',
    'QC': 'Quebec',
    'AB': 'Alberta',
    'SK': 'Saskatchewan',
    'MB': 'Manitoba',
    'NU': 'Nunavut'
}
df['node_loc'] = df['node_loc'].replace(region_rename_map)

In [28]:
platts_tecs_exclude = False

In [29]:
# --------------------
# Add data to scenario
# --------------------
with scen.transact("Calibrate historical_new_capacity to PLATTS."):
        if platts_tecs_exclude:
            df = df[~df.technology.isin(platts_tecs_exclude)]
        if remove_orig:
            filters = {
                "technology": df.technology.unique(),
                "year_vtg": df.year_vtg.unique(),
                "node_loc": df.node_loc.unique(),
            }
            for par in [
                "historical_new_capacity",
                "bound_new_capacity_lo",
                "bound_new_capacity_up",
                "growth_new_capacity_lo",
                "initial_new_capacity_lo",
                "growth_new_capacity_up",
                "initial_new_capacity_up",
            ]:
                rem = scen.par(par, filters=filters)
                rem = (
                    rem.merge(
                        df, how="inner", on=["technology", "node_loc", "year_vtg"]
                    )
                    .drop(["unit_y", "value_y"], axis=1)
                    .rename(columns={"value_x": "value", "unit_x": "unit"})
                )
                scen.remove_par(par, rem)
            filters = {
                "technology": df.technology.unique(),
                "year_act": df.year_vtg.unique(),
                "node_loc": df.node_loc.unique(),
            }
            for par in [
                "bound_total_capacity_lo",
                "bound_total_capacity_up",
            ]:
                rem = scen.par(par, filters=filters)
                rem = (
                    rem.merge(
                        df.rename(columns={"year_vtg": "year_act"}),
                        how="inner",
                        on=["technology", "node_loc", "year_act"],
                    )
                    .drop(["unit_y", "value_y"], axis=1)
                    .rename(columns={"value_x": "value", "unit_x": "unit"})
                )
                scen.remove_par(par, rem)
        if verbose:
            print("Populating parameter `historical_new_capacity`")
        df = df[df["year_vtg"] > 1955]
        hist_cap = df[df["year_vtg"] < scen.firstmodelyear]
        scen.add_par("historical_new_capacity", hist_cap)

        scen.add_par("bound_new_capacity_lo", df)
        df.loc[df["year_vtg"] >= scen.firstmodelyear, "value"] *= 1.005
        scen.add_par("bound_new_capacity_up", df)

        if add_total_capacity_lo is not None:
            cumcap = pd.DataFrame()
            tmp = df.sort_values(
                by=["node_loc", "technology", "year_vtg", "unit"]
            ).set_index(["node_loc", "technology", "year_vtg", "unit"])

            # Because the values were original foru a bound_new_capacity_XX, the values have been divided
            # by the duration period, hence the capacity values for bound_total_capacity_lo need to be
            # multiplied by the duration period.
            period_duration = (
                scen.par("duration_period")
                .rename(columns={"year": "year_vtg"})
                .set_index("year_vtg")
            )
            tmp.value *= period_duration["value"]
            tmp = tmp.reset_index()

            for node in tmp.node_loc.unique().tolist():
                if add_total_capacity_lo_tec == "all":
                    tecs = tmp.loc[tmp.node_loc == node].technology.unique()
                else:
                    tecs = add_total_capacity_lo_tec
                for t in tecs:
                    add_data = (
                        tmp.loc[(tmp.node_loc == node) & (tmp.technology == t)]
                        .copy()
                        .set_index(["node_loc", "technology", "year_vtg", "unit"])
                        .cumsum()
                    )
                    cumcap = pd.concat([cumcap, add_data])
            cumcap = cumcap.reset_index().rename(columns={"year_vtg": "year_act"})
            if type(add_total_capacity_lo) == list:
                cumcap = cumcap.loc[cumcap.year_act.isin(add_total_capacity_lo)]
            scen.add_par("bound_total_capacity_lo", cumcap)


Populating parameter `historical_new_capacity`


In [30]:
df_copy = df

In [31]:
df.to_csv('historical_capacity_by_province.csv')

In [32]:
## Now moving to Activity Calibration

In [33]:
path_activity_IEA2022 = '/Users/muhammadawais/Documents/GitHub/message-ix-canada/data/data_country/activity/REV2024_allISO_IEA_V2.parquet'

In [34]:
data_iea = pd.read_parquet(path_activity_IEA2022)

In [35]:
%reload_ext autoreload

In [36]:
from model.utlities.calibrate_to_IEA_offline import main as calibrate_to_IEA_offline
import warnings
from itertools import product

import pandas as pd
import yaml

from model.utlities.compile_IEA_data_offline import CopyHistAct
from model.utlities.get_historical_years import main as get_historical_years

from model.utlities.adjust_lifetime_to_history import main as adjust_lifetime_to_history

from model.utlities.adjust_cf_to_history import main as adjust_cf_to_history

In [37]:
ctx = 1
verbose = False

In [38]:
scenario = msg_scenario

In [40]:
calibration_year = 2015

In [41]:
scheme="MESSAGEix-GLOBIOM"

In [42]:
time_step = 5

years = get_historical_years(scenario)
calibration_year = (
    max(years) if calibration_year is None else calibration_year
)
years = [
    y for y in scenario.set("year") if y >= min(years) and y <= calibration_year
]
print(">>> Activity Calibration for: ", years)
path_activity_IEA2022 = '/Users/muhammadawais/Documents/GitHub/message-ix-canada/data/data_country/activity/REV2024_allISO_IEA_V2.parquet'
path_activity_IEA_mapping = '/Users/muhammadawais/Documents/GitHub/message-ix-canada/data/data_country/activity/IEA_mappings.csv'
# Read data required for processing
data_iea = pd.read_parquet(path_activity_IEA2022)

# Retrieve IEA mapping file
data_tec = pd.read_csv(path_activity_IEA_mapping)
# Filter out correct scheme
data_tec = data_tec.loc[data_tec["scheme"] == scheme]
data_tec["IEA flow"] = data_tec["IEA flow"].str.split(", ")
data_tec["IEA product"] = data_tec["IEA product"].str.split(", ")
# Technologies included in the data analysis
data_tec = data_tec[data_tec["include"] == "y"].dropna(subset=["technology"])
# Multiple technologies to one IEA flow-product
data_tec_share = [x for x in data_tec["technology"].tolist() if "," in x]

# Read historic share data

path_historic_elec_rc_shares = '/Users/muhammadawais/Documents/GitHub/message-ix-canada/data/data_country/demand/rc_sector/historic_specifc_electricity_share.csv'
data_historic_elec_rc_shares = pd.read_csv(path_historic_elec_rc_shares)

# Read various scenario paramters to avoid re-reading from scenario object.
data_par = {}
data_par["input"] = scenario.par("input")
data_par["output"] = scenario.par("output")
data_par["hist_new_cap"] = scenario.par("historical_new_capacity")
data_par["bound_new_capacity_lo"] = scenario.par("bound_new_capacity_lo")
data_par["capacity_factor"] = scenario.par("capacity_factor")
data_par["duration_period"] = scenario.par("duration_period")
data_par["technical_lifetime"] = scenario.par("technical_lifetime")

# Filter out range of years for which data is required.
data_iea = data_iea.loc[(data_iea["TIME"] >= 1990) & (data_iea["TIME"] <= 2020)]


>>> Activity Calibration for:  [1990, 1995, 2000, 2005, 2010, 2015]


In [43]:
region = scenario.set('node')[2:]

In [44]:
data_par['hist_new_cap']

,node_loc,technology,year_vtg,value,unit
0,Alberta,biomass_t_d,2015,0.394811,GW/a
1,Alberta,coal_t_d,2015,0.375790,GW/a
2,Alberta,elec_exp,2015,-0.023973,GW/a
3,Alberta,elec_t_d,2015,1.324316,GW/a
4,Alberta,foil_t_d,2015,0.003888,GW/a
...,...,...,...,...,...
2374,Yukon,wind_ppl,1995,0.000000,GW
2375,Yukon,wind_ppl,2000,0.000132,GW
2376,Yukon,wind_ppl,2005,0.000000,GW
2377,Yukon,wind_ppl,2010,0.000000,GW


In [45]:
tecs = data_par['hist_new_cap']['technology'].unique()

In [46]:
df

,node_loc,technology,year_vtg,value,unit
1,Alberta,bio_ppl,1960,0.000000,GW
2,Alberta,bio_ppl,1965,0.000000,GW
3,Alberta,bio_ppl,1970,0.000000,GW
4,Alberta,bio_ppl,1975,0.000000,GW
5,Alberta,bio_ppl,1980,0.000000,GW
...,...,...,...,...,...
2543,Yukon,wind_ppl,2000,0.000132,GW
2544,Yukon,wind_ppl,2005,0.000000,GW
2545,Yukon,wind_ppl,2010,0.000000,GW
2546,Yukon,wind_ppl,2015,0.000000,GW


In [47]:
msg_scenario

In [61]:
if data_par == None or "historical_new_capacity" not in data_par:
    df_cap = scenario.par(
        "historical_new_capacity", {"node_loc": region, "technology": tecs}
    )
else:
    df_cap = data_par["historical_new_capacity"].copy()
    df_cap = df_cap.loc[
        (df_cap.node_loc == region) & (df_cap.technology.isin(tecs))
    ]


In [48]:
# Assign technologies for which capacities were calibrated.
platts_tecs = df.technology.drop_duplicates()
platts_tecs = platts_tecs.tolist()  

# Change Technology Lifetime
par_exclude = [
    x
    for x in msg_scenario.par_list()
    if any(y in x for y in ["bound_", "historical_", "ref_", "emission_factor"])
]

par_exclude_hist = par_exclude + [
    x
    for x in msg_scenario.par_list()
    if any(y in x for y in ["growth_", "initial_"])
]

In [49]:
# Adjust historical lifetimes across technologies, so that the earliest
# vintage is still active until 2025. Later vintages, pre 2020, are
# adjusted to slowly reduce to original
# lifetime.
adjust_lifetime_to_history(
    msg_scenario,
    par_exclude=par_exclude_hist,
    tecs= platts_tecs,
)

bio_ppl Canada
No historical capacity parameterized for bio_ppl in Canada. Continuing.
coal_adv Canada
No historical capacity parameterized for coal_adv in Canada. Continuing.
coal_ppl Canada
No historical capacity parameterized for coal_ppl in Canada. Continuing.
coal_ppl_u Canada
No historical capacity parameterized for coal_ppl_u in Canada. Continuing.
foil_ppl Canada
No historical capacity parameterized for foil_ppl in Canada. Continuing.
gas_cc Canada
No historical capacity parameterized for gas_cc in Canada. Continuing.
gas_ct Canada
No historical capacity parameterized for gas_ct in Canada. Continuing.
gas_ppl Canada
No historical capacity parameterized for gas_ppl in Canada. Continuing.
hydro_hc Canada
No historical capacity parameterized for hydro_hc in Canada. Continuing.
hydro_lc Canada
No historical capacity parameterized for hydro_lc in Canada. Continuing.
loil_ppl Canada
No historical capacity parameterized for loil_ppl in Canada. Continuing.
nuc_lc Canada
No historical c

In [60]:
df = msg_scenario.par('historical_new_capacity')

In [56]:
print(scenario.par("technical_lifetime", filters={"node_loc": node, "technology": tec}))


Empty DataFrame
Columns: [node_loc, technology, year_vtg, value, unit]
Index: []


In [55]:
tec = 'wind_ppl'

In [127]:
msg_scenario.commit('')

In [128]:
caseName = msg_scenario.model + '__' + msg_scenario.scenario + '__v' + str(msg_scenario.version)
msg_scenario.solve(model='MESSAGE', case=caseName, solve_options={'lpmethod': '4'})

--- Warning: The GAMS version [45.1.0] differs from the API version [24.8.3].
--- Job MESSAGE_run.gms Start 01/29/25 17:03:15 45.1.0 88bbff72 DEX-DEG x86 64bit/macOS
--- Applying:
    /Library/Frameworks/GAMS.framework/Versions/45/Resources/gmsprmun.txt
--- GAMS Parameters defined
    Input /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/MESSAGE_run.gms
    ScrDir /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/225a/
    SysDir /Library/Frameworks/GAMS.framework/Versions/45/Resources/
    LogOption 4
    --in /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/data/MsgData_MESSAGEix-CA__baseline__v28.gdx
    --out /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgOutput_MESSAGEix-CA__baseline__v28.gdx
    --iter /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgIterationReport_MESSAGEix-CA__baseline__v28.gdx
Licensee: Small MUD - 5 User License                     S220901|0002AP-GEN
  

ModelError: GAMS errored with return code 3:
    There was an execution error

For details, see the terminal output above, plus:
Listing   : /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/MESSAGE_run.lst
Log file  : /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/MESSAGE_run.log
Input data: /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/data/MsgData_MESSAGEix-CA__baseline__v28.gdx

In [50]:


# Manually reset all the capacity factors to the original "SSP"
# values. This is needed, because the IEA data uses the capacity
# factors in order to derive the activity split if more than one
# technology is mapped to a single flow/product.
df = pd.read_csv(f'/Users/muhammadawais/Documents/GitHub/message-ix-canada/data/powerplant_capacity/SSP_original_cf.csv')


In [51]:
df

,node_loc,technology,value,unit
0,R12_AFR,bio_istig,0.850,%
1,R12_CHN,bio_istig,0.850,%
2,R12_EEU,bio_istig,0.850,%
3,R12_FSU,bio_istig,0.850,%
4,R12_LAM,bio_istig,0.850,%
...,...,...,...,...
187,R12_RCPA,hydro_lc,0.448,%
188,R12_RCPA,loil_cc,0.850,%
189,R12_RCPA,loil_ppl,0.650,%
190,R12_RCPA,nuc_hc,0.850,%


In [53]:
df_cf = msg_scenario.par('capacity_factor')

In [54]:
import pandas as pd

# Sample DataFrames (replace these with your actual DataFrames)
bas_msgca_cf = df_cf

base_cf = df

# Extract technology and value columns from bas_msgca_cf
# Extract technology and value columns from base_cf
technology_values = base_cf.set_index("technology")["value"]

# Iterate through each technology in bas_msgca_cf and replace values
for idx, row in bas_msgca_cf.iterrows():
    matching_rows = base_cf[base_cf["technology"] == row["technology"]]
    if not matching_rows.empty:
        bas_msgca_cf.at[idx, "value"] = matching_rows.iloc[0]["value"]

print(bas_msgca_cf)

       node_loc technology  year_vtg  year_act  time  value unit
0       Alberta    CF4_TCE      2010      2010  year    1.0    -
1       Alberta    CF4_TCE      2015      2015  year    1.0    -
2       Alberta    CF4_TCE      2020      2020  year    1.0    -
3       Alberta    CF4_TCE      2025      2025  year    1.0    -
4       Alberta    CF4_TCE      2030      2030  year    1.0    -
...         ...        ...       ...       ...   ...    ...  ...
144692    Yukon   wind_ppl      2050      2055  year    0.3    -
144693    Yukon   wind_ppl      2050      2060  year    0.3    -
144694    Yukon   wind_ppl      2055      2055  year    0.3    -
144695    Yukon   wind_ppl      2055      2060  year    0.3    -
144696    Yukon   wind_ppl      2060      2060  year    0.3    -

[144697 rows x 7 columns]


In [57]:
msg_scenario.check_out()

In [58]:
msg_scenario.add_par("capacity_factor", bas_msgca_cf)

In [139]:
bas_msgca_cf

,node_loc,technology,year_vtg,year_act,time,value,unit
0,Alberta,CF4_TCE,2010,2010,year,1.0,-
1,Alberta,CF4_TCE,2015,2015,year,1.0,-
2,Alberta,CF4_TCE,2020,2020,year,1.0,-
3,Alberta,CF4_TCE,2025,2025,year,1.0,-
4,Alberta,CF4_TCE,2030,2030,year,1.0,-
...,...,...,...,...,...,...,...
145108,Yukon,wind_ppl,2050,2055,year,0.3,-
145109,Yukon,wind_ppl,2050,2060,year,0.3,-
145110,Yukon,wind_ppl,2055,2055,year,0.3,-
145111,Yukon,wind_ppl,2055,2060,year,0.3,-


In [186]:
msg_scenario.check_out()

AttributeError: 'ellipsis' object has no attribute 'check_out'

In [121]:
df_cf = df[df['node_loc'] == 'R12_NAM']

In [127]:
df_cf_msg.to_csv('base_msgca_cf.csv')

In [125]:
df_cf_msg = msg_scenario.par('capacity_factor')

In [59]:
calibration_year = msg_scenario.firstmodelyear

In [145]:
calibration_year

2020

In [146]:
ctx.platts_tecs

AttributeError: 'int' object has no attribute 'platts_tecs'

In [60]:
from model.utlities.utilities import calc_real_cf, get_cum_inst_cap, closest

In [61]:
regions = [
            r for r in msg_scenario.set("node") if not any(n in r for n in ["World", "Canada"])
        ]

In [185]:
msg_scenario

Ellipsis

In [62]:
use_reference_cf_data = False

In [63]:
convergence_year=2040
max_adj=0.25
upper_max=0.95
tolerance=0.025
hist_constant=False

In [64]:
platts_tecs = msg_scenario.par('historical_new_capacity')['technology'].unique()

In [73]:
years = (
        scen.par(
            "historical_new_capacity", filters={"node_loc": [node], "technology": [tec]}
        )[["year_vtg", "value"]]
        .set_index("year_vtg")
        .to_dict()["value"]
    )

In [75]:
calibration_year

2020

In [86]:
def get_cum_inst_cap(scen, node, tec, calibration_year=None):
    # Retrieve `historical_new_capacity` parameter.
    years = (
        scen.par(
            "historical_new_capacity", filters={"node_loc": [node], "technology": [tec]}
        )[["year_vtg", "value"]]
        .set_index("year_vtg")
        .to_dict()["value"]
    )
    
    # Retrieve calibrated new capacity for model years
    if calibration_year is not None and calibration_year >= scen.firstmodelyear:
        years2 = scen.par(
            "bound_new_capacity_lo", filters={"node_loc": [node], "technology": [tec]}
        )[["year_vtg", "value"]]
        filter_years = [
            y
            for y in years2.year_vtg.unique()
            if y not in years.keys() and y <= calibration_year
        ]
        years2 = (
            years2.loc[years2.year_vtg.isin(filter_years)]
            .set_index("year_vtg")
            .to_dict()["value"]
        )
        years.update(years2)

    # Filter out all non-zero data
    years = {y: years[y] for y in years if years[y] > 0.0}

    dfs = []
    for y in years:
        # Check year_act and duration_period for each year before creating the dataframe
        year_act = scen.years_active(node, tec, y)
        duration_value = scen.par("duration_period", filters={"year": y}).iloc[0].value

        if not year_act or pd.isna(duration_value):
            print(f"Skipping year {y} due to missing year_act or duration_value.")
            continue

        # Prepare the dataframe and pivot table
        df = pd.DataFrame(
            {
                "year_vtg": y,
                "year_act": year_act,
                "value": years[y] * float(duration_value),
            }
        )
        df_pivot = df.pivot_table(index="year_vtg", columns="year_act", values="value")
        dfs.append(df_pivot)

    if dfs:
        df = pd.concat(dfs)
    else:
        df = pd.DataFrame()

    return df


In [87]:
 df = get_cum_inst_cap(msg_scenario, node, tec, calibration_year)

KeyError: 0

In [106]:
tec

'coal_ppl_u'

In [ ]:
path_UE_input

In [112]:
# --------------------------------------------------------------------------------------------------------
# Set `total_capacity_lo` for selected technologies to ensure that the expected capacities are not retired
# --------------------------------------------------------------------------------------------------------

add_lower_total_cap = {
    "nuc_lc": 2035,
    "nuc_hc": 2035,
    "hydro_lc": 2025,
    "hydro_hc": 2025,
    "coal_adv": 2025,  # TEMPORARY
}

# Retrieve historical new capacity
df_hist = msg_scenario.par(
    "historical_new_capacity",
    filters={"technology": [t for t in add_lower_total_cap.keys()]},
)

# Retrieve bound_new_capacity_lo, filtering out any hsaitoric periods
df = msg_scenario.par(
    "bound_new_capacity_lo",
    filters={"technology": [t for t in add_lower_total_cap.keys()]},
)
df = df.loc[~df.year_vtg.isin(df_hist.year_vtg.unique().tolist())]

# Merge dataframes
df = pd.concat([df, df_hist])
df = (
    df.loc[df.value > 0]
    .sort_values(by=["node_loc", "technology", "year_vtg"])
    .reset_index()
    .drop("index", axis=1)
)


In [113]:
# Multiply values by duration period as new cpacity is given in annual terms
duration = (
    msg_scenario.par("duration_period")
    .rename(columns={"year": "year_vtg"})
    .set_index("year_vtg")
)
df = df.set_index(["node_loc", "technology", "year_vtg"])
df.value *= duration.value
df = df.reset_index()

# Check for each vintage if the capacity will still be active
for i in df.index:
    node = df.iloc[i].node_loc
    technology = df.iloc[i].technology
    year_vtg = df.iloc[i].year_vtg
    try:
        df.loc[i, "max_year_act"] = max(
            msg_scenario.vintage_and_active_years(
                (node, technology, year_vtg), in_horizon=False
            ).year_act
        )
    except:
        test = 1
    df.loc[i, "max_bnd_yr"] = add_lower_total_cap[technology]


In [126]:


for y in [y
          for y in df.year_vtg.unique().tolist()
          if y >= msg_scenario.firstmodelyear
          ]:
        # Filter out all vintages smaller than the target year which have activity in or beyond that year
        tmp = df.loc[
            (df.year_vtg <= y) & (df.max_year_act >= y) & (y <= df.max_bnd_yr)
        ].drop(["max_year_act", "year_vtg", "unit", "max_bnd_yr"], axis=1)
        tmp = (
            tmp.groupby(["node_loc", "technology"])
            .sum(numeric_only=True)
            .assign(year_act=y, unit="GW")
            .reset_index()
        )

        msg_scenario.add_par("bound_total_capacity_lo", tmp)
        tmp.value *= 1.005
msg_scenario.add_par("bound_total_capacity_up", tmp)

In [125]:
msg_scenario.check_out()

In [ ]:
calibrate_UE_gr_to_demand(
    msg_scenario,
    ctx.path_UE_input,
    ssp=self.ssp_name,
    verbose=self.verbose,
)

# ------------------------------------------------
# Step 8.5 - Add growth rates for useful-energy
# ------------------------------------------------

add_UE_share_constraints(
    self.scen,
    path_UE_share_input=ctx.path_UE_share_input,
    ssp="SSP@",
    start_year={
        "i_feed": 2025,
        "i_spec": 2025,
        "i_therm": 2025,
        "non-comm": 2030,
        "rc_spec": 2030,
        "rc_therm": 2030,
        "transport": 2030,
    },
    calibration_year=2015,
    clean_relations=True,
    verbose=self.verbose,
)


In [129]:
msg_scenario.to_excel("/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/scenarios/baseline/model_default_v12.xlsx")

In [92]:
msg_scenario.commit('')